In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [3]:
data = pd.read_csv(r"D:\ansci-4040-fall-2026\jl4937_6040-project-1\dataset\Data_set_prep_assignment_1.csv")

In [9]:
duplicate_count = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 60653


In [4]:
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,NaN,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,NaN,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


In [5]:
missing_values = data.isnull().sum()
print(missing_values)

AnimalId                  1701003
LactationNumber           1701003
DaysInMilk                1701007
ReproductionStatus        1701003
EventDate                       0
Avgmilkflow                   172
Flow30_60Session                0
YieldFirst2Min_Session          0
YieldSession                    0
DurationSession_sec             0
milking                         0
dtype: int64


In [7]:
# Split the data into two groups: Full value and With missing value
data_complete = data.dropna()        # Full datarow （Used for training）
data_missing = data[data.isnull().any(axis=1)]  # Missing value rows （For prediction）

print(f"Full data row count: {len(data_complete)}")
print(f"Missing value row count: {len(data_missing)}")

# 处理ReproductionStatus的mixed type问题（转成数字编码）
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_complete['ReproductionStatus_encoded'] = le.fit_transform(
    data_complete['ReproductionStatus'].astype(str)
)

# Check what categories are in ReproductionStatus
print(data_complete['ReproductionStatus'].value_counts())

# Check the encoded values of ReproductionStatus
print(data_complete['ReproductionStatus_encoded'].value_counts())

Full data row count: 6794283
Missing value row count: 1701138
ReproductionStatus
Pregnant    3357202
Bred        1844934
Fresh        969732
Open         622415
Name: count, dtype: int64
ReproductionStatus_encoded
3    3357202
0    1844934
1     969732
2     622415
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# ① 定义特征X 和 目标y
# 用没有缺失的6列作为特征
features = ['Avgmilkflow', 'Flow30_60Session', 
            'YieldFirst2Min_Session', 'YieldSession', 
            'DurationSession_sec', 'milking']

X = data_complete[features]
y = data_complete['DaysInMilk']

# ② Split：80%训练，20%测试
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ③ 训练模型
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ④ 用测试集评估模型好不好
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"模型误差 MAE: {mae:.2f} 天")

# ⑤ 用模型预测缺失值
X_missing = data_missing[features]
predicted_values = model.predict(X_missing)
print(predicted_values[:10])  # 查看前10个预测结果

KeyboardInterrupt: 